# CRISP-DM Phase 4 — Modeling

**Project:** ML4B SoSe 2026 — Gym Exercise Recognition from Apple Watch Sensor Data  
**Dataset:** RecoFit (Microsoft Research, Morris et al., CHI 2014)  
**Phase:** 4 of 6 — Modeling  

## Goal
Train and compare three classical ML classifiers on the 47-feature matrix produced in Phase 3:
- **Random Forest** — baseline; robust tabular learner with feature importance
- **XGBoost** — gradient boosting; typically outperforms RF on tabular data
- **SVM (RBF kernel)** — classical HAR benchmark for comparison with related work

See `docs/decisions/ADR-009-model-selection-rationale.md` for the full selection rationale.

## Primary Metric
**Macro-averaged F1** — NOT accuracy. The validation and test sets retain the original class
distribution where `rest` dominates (~89% of windows). Accuracy would reward a model that
only predicts `rest`. Macro F1 penalises ignoring any of the 6 exercise classes equally.

## Inputs
| File | Size | Description |
|------|------|-------------|
| `data/processed/train_features.csv` | 21,490 windows | Balanced (undersampled rest class) |
| `data/processed/val_features.csv` | ~13,888 windows | Original distribution — used for model selection |
| `data/processed/test_features.csv` | ~30,096 windows | Original distribution — **used ONLY in Phase 5** |

## Outputs
- Trained model serialised to `models/saved/<model_name>.joblib`
- Confusion matrix plots saved to `reports/figures/`
- Feature importance plot saved to `reports/figures/`

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# find_project_root() locates pyproject.toml by walking up from CWD.
# This makes the notebook runnable from any directory — not just the project root.
from ml4b.utils.config import DATA_PROCESSED, MODELS_DIR, REPORTS_DIR, find_project_root
from ml4b.models.train import train_random_forest, train_svm, train_xgboost
from ml4b.models.evaluate import compare_models, evaluate_model, save_model

PROJECT_ROOT = find_project_root()

print(f'Python:        {sys.version.split()[0]}')
print(f'Project root:  {PROJECT_ROOT}')
print(f'DATA_PROCESSED: {DATA_PROCESSED}')
print(f'MODELS_DIR:    {MODELS_DIR}')
print(f'REPORTS_DIR:   {REPORTS_DIR}')

## 2. Load Data

We load the feature matrices produced in Phase 3. Each CSV contains:
- **Metadata columns** (not features): `subject_id`, `exercise_name`, `window_id`
- **Feature columns**: 47 numeric features (7 per-axis statistics × 6 axes + 3 magnitudes + 2 FFT)

We separate features (`X`) from labels (`y`) before training. The test set is loaded for shape
inspection only — it is NOT used in this notebook; evaluation on the test set happens in Phase 5.

In [ ]:
# Non-feature columns that identify context but are not model inputs
META_COLS = ['subject_id', 'exercise_name', 'window_id']
LABEL_COL = 'exercise_name'

# Load feature CSVs produced by notebooks/03_data_preparation.ipynb
train_df = pd.read_csv(DATA_PROCESSED / 'train_features.csv')
val_df   = pd.read_csv(DATA_PROCESSED / 'val_features.csv')
test_df  = pd.read_csv(DATA_PROCESSED / 'test_features.csv')  # inspected, NOT used for fitting

# Separate feature matrix X from label vector y
# drop META_COLS to get purely numeric features in consistent column order
X_train = train_df.drop(columns=META_COLS).values
y_train = train_df[LABEL_COL].values

X_val = val_df.drop(columns=META_COLS).values
y_val = val_df[LABEL_COL].values

X_test = test_df.drop(columns=META_COLS).values  # shape check only
y_test = test_df[LABEL_COL].values

# Determine class names from training labels for consistent ordering across plots
CLASS_NAMES = sorted(train_df[LABEL_COL].unique().tolist())
FEATURE_NAMES = [c for c in train_df.columns if c not in META_COLS]

print('Training set:  ', X_train.shape, '| Classes:', dict(pd.Series(y_train).value_counts()))
print('Validation set:', X_val.shape,   '| Classes:', dict(pd.Series(y_val).value_counts()))
print('Test set:      ', X_test.shape,  '| (NOT used in Phase 4)')
print()
print('Classes (sorted):', CLASS_NAMES)
print('N features:      ', len(FEATURE_NAMES))

## 3. Train Models

### 3.1 Random Forest (Baseline)

Random Forest is the baseline model because it:
- Handles multi-class natively with no feature scaling required
- Provides feature importance scores for interpretability
- Is robust to correlated features via random subspace selection

Key hyperparameters: `n_estimators=200`, `class_weight='balanced'`.
See `src/ml4b/models/train.py` and ADR-009 for full rationale.

In [ ]:
t0 = time.time()
rf_model = train_random_forest(X_train, y_train)
rf_time = time.time() - t0
print(f'Random Forest trained in {rf_time:.1f}s')

### 3.2 XGBoost

XGBoost uses sequential boosting — each tree corrects the residuals of the previous ensemble.
This often outperforms Random Forest on tabular data by trading some variance for better
bias reduction. Class imbalance is handled via `compute_sample_weight('balanced')`
because XGBoost does not accept `class_weight=` directly.

Key hyperparameters: `n_estimators=300`, `learning_rate=0.1`, `max_depth=6`.
See `src/ml4b/models/train.py` and ADR-009.

In [ ]:
t0 = time.time()
xgb_model = train_xgboost(X_train, y_train)
xgb_time = time.time() - t0
print(f'XGBoost trained in {xgb_time:.1f}s')

### 3.3 SVM (RBF kernel)

SVM with a radial basis function kernel is a classical baseline in HAR (Human Activity
Recognition) literature, enabling comparison with related work. SVMs are sensitive to
feature scale, so `StandardScaler` is applied inside a sklearn `Pipeline` — this guarantees
that the same scaling is applied automatically at prediction time in the Streamlit app.

> **Note:** SVM training with `probability=True` (Platt scaling) on ~21,000 samples can
> take 5–15 minutes. This is expected.

In [ ]:
t0 = time.time()
svm_model = train_svm(X_train, y_train)
svm_time = time.time() - t0
print(f'SVM trained in {svm_time:.1f}s')

print(f'\nTraining time summary:')
print(f'  Random Forest: {rf_time:.1f}s')
print(f'  XGBoost:       {xgb_time:.1f}s')
print(f'  SVM:           {svm_time:.1f}s')

## 4. Evaluate Models on Validation Set

We evaluate all three models on the **validation set** (original class distribution).
Primary metric: **macro F1**. Accuracy is shown for context only.

Confusion matrices are saved to `reports/figures/` for visual inspection.

In [ ]:
FIGURES_DIR = REPORTS_DIR  # reports/figures/
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Evaluate each model on the validation set and collect standardised result dicts
rf_result  = evaluate_model(rf_model,  X_val, y_val, 'Random Forest', CLASS_NAMES, FIGURES_DIR)
xgb_result = evaluate_model(xgb_model, X_val, y_val, 'XGBoost',       CLASS_NAMES, FIGURES_DIR)
svm_result = evaluate_model(svm_model, X_val, y_val, 'SVM',           CLASS_NAMES, FIGURES_DIR)

for result in [rf_result, xgb_result, svm_result]:
    print(f"--- {result['model_name']} ---")
    print(f"  Accuracy:  {result['accuracy']:.4f}")
    print(f"  Macro F1:  {result['macro_f1']:.4f}  ← PRIMARY METRIC")
    print(f"  Per-class F1:")
    for cls, f1 in result['per_class_f1'].items():
        print(f"    {cls:<20} {f1:.4f}")
    print()

## 5. Model Comparison

The comparison table sorts all models by macro F1 descending.
The model in the first row is selected as the best model for Phase 5 test evaluation.

In [ ]:
results = [rf_result, xgb_result, svm_result]
comparison_df = compare_models(results)

print('Model comparison (sorted by macro F1 ↓):')
print(comparison_df.to_string(index=False))

best_model_name = comparison_df.iloc[0]['model_name']
best_macro_f1   = comparison_df.iloc[0]['macro_f1']
print(f'\nBest model: {best_model_name} (macro F1 = {best_macro_f1:.4f})')

## 6. Confusion Matrices

Row-normalised confusion matrices show, for each true class, what fraction of windows
were predicted as each class. This makes it easy to spot systematic confusions
(e.g. lateral_raise confused with shoulder_press) regardless of class size.

Saved to `reports/figures/confusion_matrix_<model>.png`.

In [ ]:
# Confusion matrices were already saved to FIGURES_DIR inside evaluate_model().
# Here we display them inline for the notebook record.

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

import seaborn as sns
from sklearn.metrics import confusion_matrix

for ax, result in zip(axes, results):
    cm = result['confusion_matrix']
    # Row-normalise so each class is on equal footing regardless of size
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        vmin=0, vmax=1, ax=ax
    )
    ax.set_title(f"{result['model_name']}\nmacro F1 = {result['macro_f1']:.4f}")
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Confusion Matrices — Validation Set (row-normalised)', fontsize=14, y=1.02)
plt.tight_layout()

# Save the combined figure as an additional overview
fig.savefig(FIGURES_DIR / 'confusion_matrices_all_models.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved combined confusion matrix plot to {FIGURES_DIR}')

## 7. Select and Save Best Model

The model with the highest macro F1 on the validation set is selected and serialised
to `models/saved/` using `joblib.dump`. This file is loaded by the Streamlit app and
used as-is for final test set evaluation in Phase 5.

We also print the full classification report of the best model for the notebook record.

In [ ]:
# Map model name to the trained object
model_registry = {
    'Random Forest': rf_model,
    'XGBoost':       xgb_model,
    'SVM':           svm_model,
}
best_result = next(r for r in results if r['model_name'] == best_model_name)
best_model  = model_registry[best_model_name]

# Save the best model to models/saved/
MODELS_DIR.mkdir(parents=True, exist_ok=True)
saved_path = save_model(best_model, best_model_name, MODELS_DIR)
print(f'Best model saved to: {saved_path}')

# Also save a copy named 'best_model' for the Streamlit app to load by convention
import joblib
best_model_path = MODELS_DIR / 'best_model.joblib'
joblib.dump(best_model, best_model_path)
print(f'Streamlit-ready copy saved to: {best_model_path}')

print(f'\nFull classification report for {best_model_name}:')
print(best_result['classification_report'])

## 8. Feature Importance

For tree-based models (Random Forest, XGBoost), we extract feature importances to
understand which sensor axes and statistics drive classification decisions.

High-importance features validate that the feature engineering in Phase 3 captured
the right motion patterns. If unexpected features dominate, it may indicate leakage
or an engineering issue to revisit.

In [ ]:
def plot_feature_importance(model, feature_names, model_name, save_dir, top_n=20):
    """Plot top-N feature importances for a tree-based model."""
    # Random Forest exposes .feature_importances_ directly.
    # XGBClassifier also exposes .feature_importances_ (gain-based by default).
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    else:
        print(f'{model_name} does not expose feature_importances_ — skipping.')
        return

    # Sort features by importance descending and keep top N
    indices = np.argsort(importances)[::-1][:top_n]
    top_features   = [feature_names[i] for i in indices]
    top_importance = importances[indices]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top_features[::-1], top_importance[::-1], color='steelblue')
    ax.set_xlabel('Feature Importance (mean decrease in impurity)')
    ax.set_title(f'Top {top_n} Feature Importances — {model_name}')
    plt.tight_layout()

    safe_name = model_name.lower().replace(' ', '_')
    save_path = save_dir / f'feature_importance_{safe_name}.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {save_path}')


# Always plot RF and XGBoost importances for comparison
plot_feature_importance(rf_model,  FEATURE_NAMES, 'Random Forest', FIGURES_DIR)
plot_feature_importance(xgb_model, FEATURE_NAMES, 'XGBoost',       FIGURES_DIR)

# SVM does not have feature_importances_ — this is expected and handled gracefully
plot_feature_importance(svm_model, FEATURE_NAMES, 'SVM', FIGURES_DIR)

## Summary

> **Fill in after running the notebook end-to-end.**

### Best Model
| | |
|---|---|
| Model | *(fill after run)* |
| Macro F1 (val) | *(fill after run)* |
| Accuracy (val) | *(fill after run)* |

### Per-class F1 on Validation Set
*(Fill after run — which exercise classes are hardest to classify and why?)*

### Key Findings from Feature Importance
*(Fill after run — which axes/statistics dominate? Does this align with the expected motion patterns?)*

### Decision for Phase 5
The best model (`models/saved/best_model.joblib`) will be evaluated on the **held-out test set**
in `notebooks/05_evaluation.ipynb`. No further model selection or tuning occurs after this point
to avoid test set contamination.